# Baseline training (ml-training-loop)

Minimal Kaggle notebook that:
1. Installs deps
2. Reads `WANDB_API_KEY` from Kaggle secrets (falling back to env)
3. Initializes a W&B run
4. Runs a tiny training loop as a placeholder
5. Saves metrics + a fake artifact to `/kaggle/working/`

Replace this with your real training. Keep the W&B init block and the final save block.

In [ ]:
%pip install --quiet wandb
import json, os, random, time
from pathlib import Path

import wandb

# Prefer Kaggle user secret, fall back to env injected from the runner
try:
    from kaggle_secrets import UserSecretsClient
    WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)

WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

In [ ]:
run = wandb.init(
    project=os.environ.get("WANDB_PROJECT", "ml-playground"),
    config={
        "lr": float(os.environ.get("LR", 1e-3)),
        "batch_size": int(os.environ.get("BATCH_SIZE", 32)),
        "epochs": int(os.environ.get("EPOCHS", 3)),
        "seed": int(os.environ.get("SEED", 42)),
    },
    reinit=True,
)

In [ ]:
# Placeholder training loop — replace with your model
for epoch in range(run.config.epochs):
    loss = max(0.05, 1.0 * (0.85 ** epoch) + random.uniform(-0.02, 0.02))
    val_f1 = min(0.99, 0.5 + 0.1 * epoch + random.uniform(-0.01, 0.01))
    wandb.log({"epoch": epoch, "train/loss": loss, "val/f1": val_f1})
    print(f"epoch={epoch} loss={loss:.4f} val_f1={val_f1:.4f}")
    time.sleep(0.2)

run.finish()

In [ ]:
summary = {
    "finished": True,
    "final_val_f1": float(val_f1),
    "config": dict(run.config),
}
(WORKING / "metrics.json").write_text(json.dumps(summary, indent=2))
print("Saved", WORKING / "metrics.json")